# **Starter code**

In [ ]:
"""
Although we provide a list of permitted packages,
not all of them are necessarily required,
nor is the list guaranteed to be sufficient.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import json
import joblib

In [ ]:
import requests

BASE_URL = "https://usaaio-round1-problem5.billowing-field-1946.workers.dev"
DATASET_TOKEN = "52c963809161730e3fe2da4248a6120f630c0d7815b48e7b36baae864211cfb6"

def download_dataset(filename):
    response = requests.get(
        f"{BASE_URL}/{filename}",
        headers={"Authorization": f"Bearer {DATASET_TOKEN}"},
        timeout=30,
    )
    response.raise_for_status()
    with open(filename, "wb") as f:
        f.write(response.content)
    return filename

train_path = download_dataset("train.json")


In [ ]:
with open(train_path, "r") as f:
    data_train = json.load(f)

print(type(data_train))
print(data_train.keys())

# John: Not recommended to convert to numpy array directly.
# Pandas has advantange doing feature engineering.
x_raw = np.array(data_train["x"], dtype=np.float32).reshape(-1, 1)
y_raw = np.array(data_train["y"], dtype=np.float32).reshape(-1, 1)

# Convert data_train from JSON to Pandas DataFrame

In [ ]:
# convert data_train into pandas dataframe and store in x_raw, y_raw
import pandas as pd
df = pd.DataFrame(data_train)
print(df.head())

# Take out X, y from df
x_raw = df[["x"]]
y_raw = df["y"]

# Test Data Split



In [ ]:
# Check the shape of x_raw and y_raw
print(x_raw.shape)
print(y_raw.shape)

# Do the train validation test split
#  - Split test data 20%, train+val 80%
# Look up the cheatsheet https://colab.research.google.com/drive/1L2P3-7s5kpsgizrZ5L9tpYTe1-djOoJT#scrollTo=o1e41qb8U9xW
from sklearn.model_selection import train_test_split
x_train_val, x_test, y_train_val, y_test = train_test_split(x_raw, y_raw, test_size=0.2, random_state=42)

# Check the shape of training validation and test data sets.
print(x_train_val.shape)
print(x_test.shape)
print(y_train_val.shape)
print(y_test.shape)



# Validation Data Split

In [ ]:
# Train validation test split
#  - Split validation data 25% of train+val, 75% training data of train+val
# Look up the cheatsheet https://colab.research.google.com/drive/1L2P3-7s5kpsgizrZ5L9tpYTe1-djOoJT#scrollTo=o1e41qb8U9xW
from sklearn.model_selection import train_test_split
x_train, x_val, y_train, y_val = train_test_split(x_train_val, y_train_val, test_size=0.25, random_state=42)

# Check the shape of training validation and test data sets.
print(x_train.shape)
print(x_val.shape)
print(y_train.shape)
print(y_val.shape)

print(x_train)

# Train the Model

In [ ]:
# MINIMUM VERSION: no feature engineering.
# The pandas DataFrame goes straight to numpy, then straight into the model.

# Convert the data you want to input into model into numpy array
x_train_np = x_train.to_numpy()
y_train_np = y_train.to_numpy()
x_val_np = x_val.to_numpy()
y_val_np = y_val.to_numpy()

# Load LinearRegression Model and train the data against it.
# Load model is included in cheatsheet
# https://colab.research.google.com/drive/1L2P3-7s5kpsgizrZ5L9tpYTe1-djOoJT#scrollTo=o1e41qb8U9xW
from sklearn.linear_model import LinearRegression
model = LinearRegression()

# train the model using training data
model.fit(x_train_np, y_train_np)

# write a FUNCTION to calculate the mean_abs_error, and mean_pct_error between
# actual value and predicted values
def mean_abs_error(actual, predicted):
  return np.abs(actual - predicted).mean()

def mean_pct_error(actual, predicted):
  return np.abs((actual - predicted) / actual).mean()

# Get the model output on training data
print("Training Result")
y_train_pred = model.predict(x_train_np)
print(mean_abs_error(y_train, y_train_pred))
print(mean_pct_error(y_train, y_train_pred))


# Get the model output on validation data
print("Validation Result")
y_val_pred = model.predict(x_val_np)
print(mean_abs_error(y_val, y_val_pred))
print(mean_pct_error(y_val, y_val_pred))

# Persist model into a joblib file
# This is the ONLY file inference needs (no scaler / onehot in the minimum version)
joblib.dump(model, "model.joblib")